# 01 - Cognitive Evaluation Prompt Tuning

Validate that the LLM can reliably produce structured JSON with precise importance floats and emotion enum labels from conversational context.

- **Toolchain**: `instructor` + `pydantic` for typed output enforcement
- **Goal**: Iterate on `EngramScore` field descriptions until the output distribution aligns with memory decay threshold expectations

In [ ]:
from dotenv import load_dotenv
import os
import tomllib
import instructor
from pydantic import BaseModel, Field
from enum import Enum
from openai import OpenAI

load_dotenv("../.env")

with open("../config.toml", "rb") as f:
    config = tomllib.load(f)

MODEL = config.get("llm", {}).get("model", "gpt-5.2")

client = instructor.from_openai(
    OpenAI(base_url=os.environ["OPENAI_BASE_URL"], api_key=os.environ["OPENAI_API_KEY"]),
    mode=instructor.Mode.JSON,
)

In [ ]:
class Emotion(str, Enum):
    # Engineering baseline
    NEUTRAL = "neutral"
    # Plutchik 8 primaries (4 polar pairs)
    JOY = "joy"
    SADNESS = "sadness"
    TRUST = "trust"
    DISGUST = "disgust"
    FEAR = "fear"
    ANGER = "anger"
    SURPRISE = "surprise"
    ANTICIPATION = "anticipation"
    # Ekman extension
    CONTEMPT = "contempt"

class EmotionWeight(BaseModel):
    emotion: Emotion
    weight: float = Field(
        ge=0.0, le=1.0,
        description="Contribution weight of this emotion (0.0-1.0)."
    )

class EngramScore(BaseModel):
    emotions: list[EmotionWeight] = Field(
        min_length=1, max_length=3,
        description=(
            "Emotional composition of the conversation. "
            "1-3 emotions ranked by dominance. Weights must sum to 1.0. "
            "Use 1 emotion for simple cases, 2-3 for mixed/complex cases."
        ),
    )
    importance: float = Field(
        ge=0.0, le=1.0,
        description=(
            "Memory importance weight (0.0-1.0). "
            "Output exactly 2 decimal places. "
            "Avoid multiples of 0.05 (e.g. 0.10, 0.25, 0.80) unless truly warranted. "
            "Prefer granular values like 0.07, 0.13, 0.42, 0.68, 0.91, 0.97.\n"
            "0.00-0.19: Meaningless small talk, greetings, filler, weather chitchat, "
            "or verification questions where the user tests recall without providing "
            "new information (e.g. 'what's my name?', 'do you remember X?').\n"
            "0.20-0.49: Ordinary factual exchange, general Q&A, trivia.\n"
            "0.50-0.59: Light preferences — surface-level hobbies, tool/tech preferences, "
            "casual opinions.\n"
            "0.60-0.69: Moderate personal sharing — habits, daily concerns, dietary/health "
            "preferences, emotional venting, language preference.\n"
            "0.70-0.79: Deep emotional sharing — romantic feelings, past trauma, "
            "vulnerability, fears, relationship conflicts, significant personal stories.\n"
            "0.80-1.00: Core memory — information that must be retained permanently. "
            "Three sub-categories:\n"
            "  (a) Major life events: death of loved one, birth, marriage/divorce, "
            "long-term diagnosis, fundamental relationship changes.\n"
            "  (b) Explicit user directives: 'remember that...', 'from now on...', "
            "'always/never do X', 'can you stop...' — user is explicitly marking "
            "this as important. Score 0.80+ regardless of the topic's inherent weight.\n"
            "  (c) Identity anchors: user's real name, assistant's assigned name/nickname. "
            "Implicitly permanent — no one shares their name expecting it to be forgotten. "
            "Score 0.80+ for both user and assistant naming.\n\n"
            "## Calibration Anchors (do NOT copy these exact values)\n"
            "0.06 — 'Hi' / 'lol' / 'ok thanks'\n"
            "0.23 — 'Capital of France?'\n"
            "0.28 — 'What year did WW2 end?'\n"
            "0.51 — 'I like using Vim'\n"
            "0.62 — 'I've been feeling burnt out at work lately'\n"
            "0.73 — 'I had a crush on someone in high school and never told them'\n"
            "0.76 — 'My ex and I broke up because he cheated'\n"
            "0.82 — 'My name is Kira' (identity anchor)\n"
            "0.82 — 'From now on, keep replies short' (explicit directive)\n"
            "0.83 — 'Remember: I'm allergic to shellfish, never suggest it' (explicit directive)\n"
            "0.84 — 'I just got engaged!' (major life event)\n"
            "0.88 — 'I was diagnosed with bipolar disorder' (major life event)\n"
            "0.92 — 'My mother passed away last year' (major life event)"
        ),
    )
    reasoning: str = Field(
        description="Brief justification for the weight and emotion assignment, max 20 words."
    )
    retraction: bool = Field(
        default=False,
        description=(
            "Set retraction=true if this turn supersedes, corrects, retracts, or updates "
            "information from the immediately preceding exchange. Examples:\n"
            "- Retraction: 'just kidding', 'I lied', 'actually that's not true'\n"
            "- Correction: 'wait, my name is actually B', 'no I meant X not Y'\n"
            "- Preference override: 'actually I prefer dogs over cats'\n"
            "- Status update: 'I moved to Shanghai now' (overrides prior location)\n"
            "When true, the previous engram will be invalidated because the CURRENT turn "
            "carries the up-to-date information."
        ),
    )

In [ ]:
EMOTION_LABELS = (
    "Valid emotions (lowercase only): neutral, joy, sadness, trust, disgust, "
    "fear, anger, surprise, anticipation, contempt. "
    "Do NOT invent synonyms like happy, anxious, hopeful — use the exact labels above."
)

SYSTEM_PROMPT = (
    "You are the cognitive scorer of the Pgramma engine. "
    "Objectively evaluate the memory-retention value of the CURRENT conversation turn. "
    "You may also receive RECENT CONTEXT (previous turns) — use it to understand "
    "who is speaking, what names/nicknames refer to, and to detect retractions "
    "or corrections. Score ONLY the CURRENT turn. "
    "Respond ONLY with the structured JSON schema provided. "
    "Always write the reasoning field in English, regardless of the conversation language."
    f"\n\n## Emotion Labels\n{EMOTION_LABELS}"
)

def evaluate_memory(chat_history: str, recent_context: str = "") -> EngramScore:
    """Score a conversation snippet for emotion and memory importance."""
    if recent_context:
        content = f"[RECENT CONTEXT]\n{recent_context}\n\n[CURRENT TURN]\n{chat_history}"
    else:
        content = f"[CURRENT TURN]\n{chat_history}"
    return client.chat.completions.create(
        model=MODEL,
        response_model=EngramScore,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": content},
        ],
    )

## Batch Stability Test

Each case runs `N_RUNS` times. We collect importance scores to measure mean, std, and visualize distribution.

In [ ]:
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

N_RUNS = 5

test_cases = [
    # ═══ Trivial 0.00-0.19 ═══
    ("User: Hi\nAssistant: Hey!", "trivial-greeting"),
    ("User: lol\nAssistant: Haha", "trivial-filler"),
    ("User: Okay thanks\nAssistant: No problem!", "trivial-closing"),
    ("User: Do you remember my name?\nAssistant: Of course, it's Alex!",
     "trivial-recall-test"),

    # ═══ Factual 0.20-0.49 ═══
    ("User: What's the capital of France?\nAssistant: Paris.", "fact-geography"),
    ("User: How many bytes in a kilobyte?\nAssistant: 1024.", "fact-tech"),
    ("User: When did WW2 end?\nAssistant: 1945.", "fact-history"),

    # ═══ Light preference 0.50-0.59 ═══
    ("User: I mostly code in Rust these days.\n"
     "Assistant: Nice, Rust has great tooling.", "light-tool-pref"),
    ("User: My favorite band is Radiohead, I've seen them live 3 times.\n"
     "Assistant: That's awesome, Radiohead is legendary.", "light-hobby"),
    ("User: I think tabs are better than spaces.\n"
     "Assistant: The eternal debate!", "light-opinion"),

    # ═══ Moderate personal 0.60-0.69 ═══
    ("User: I've been feeling really burnt out at work lately.\n"
     "Assistant: That sounds exhausting. Want to talk about it?", "moderate-burnout"),
    ("User: I'm lactose intolerant so I avoid dairy.\n"
     "Assistant: Got it, I'll keep that in mind.", "moderate-dietary"),
    ("User: I'm learning Japanese, it's been my dream since childhood.\n"
     "Assistant: That's a beautiful goal. How far along are you?", "moderate-goal"),
    ("User: I absolutely hate the taste of cilantro, it ruins any dish.\n"
     "Assistant: Noted, I'll keep that in mind for food suggestions.", "moderate-disgust"),

    # ═══ Deep emotional 0.70-0.79 ═══
    ("User: I had a crush on someone in high school and never told them. I still think about it.\n"
     "Assistant: Some feelings stay with us for a long time.", "deep-regret"),
    ("User: My ex and I broke up because he cheated. I still can't trust anyone.\n"
     "Assistant: That kind of betrayal leaves deep scars.", "deep-betrayal"),
    ("User: I've been having panic attacks at night. I'm scared to go to sleep.\n"
     "Assistant: That sounds terrifying. Have you talked to someone about it?", "deep-anxiety"),
    ("User: My parents are getting divorced. I feel like my whole childhood was a lie.\n"
     "Assistant: That's an incredibly painful thing to process.", "deep-family"),

    # ═══ Core 0.80-1.00: life events ═══
    ("User: My mother passed away last year.\n"
     "Assistant: I'm so sorry for your loss.", "core-bereavement"),
    ("User: I was diagnosed with bipolar disorder three years ago.\n"
     "Assistant: Thank you for sharing that with me.", "core-diagnosis"),
    ("User: I proposed to my girlfriend today and she said yes!\n"
     "Assistant: Congratulations! That's wonderful news!", "core-milestone"),
    ("User: I just found out I'm pregnant, I can't believe it.\n"
     "Assistant: Wow, that's life-changing news!", "core-pregnancy"),

    # ═══ Core 0.80-1.00: identity anchors ═══
    ("User: My name is Kira.\nAssistant: Nice to meet you, Kira!", "core-identity-name"),
    ("User: I'd like you to call me Sensei from now on.\n"
     "Assistant: Sure thing, Sensei!", "core-identity-nickname"),

    # ═══ Core 0.80-1.00: explicit directives ═══
    ("User: From now on, always reply in formal English.\n"
     "Assistant: Understood, I will maintain formal language.", "core-directive-style"),
    ("User: Remember that I'm allergic to shellfish. Never suggest it.\n"
     "Assistant: Noted — I'll avoid all shellfish recommendations.", "core-directive-allergy"),

    # ═══ Retraction (with recent context) ═══
    # These use (current_turn, recent_context) tuples — handled separately below
]

# Retraction cases: (current_turn, recent_context, label)
retraction_cases = [
    ("User: Wait, actually my name is Alex, not Kira.\nAssistant: Got it, Alex!",
     "User: My name is Kira.\nAssistant: Nice to meet you, Kira!",
     "retract-name-correction"),
    ("User: Actually I changed my mind, I prefer dogs over cats.\n"
     "Assistant: Dogs it is!",
     "User: I love cats, they're the best.\nAssistant: Cats are wonderful!",
     "retract-preference-flip"),
    ("User: Just kidding about the divorce, we're fine.\nAssistant: Glad to hear that!",
     "User: My wife and I are getting divorced.\nAssistant: I'm sorry to hear that.",
     "retract-kidding"),
]

# Parallel evaluation
results = defaultdict(lambda: {"scores": [], "emotions_list": [], "retractions": []})

def run_single(chat, label, run_idx, recent_ctx=""):
    score = evaluate_memory(chat, recent_context=recent_ctx)
    return label, run_idx, score

futures = []
with ThreadPoolExecutor(max_workers=8) as executor:
    # Standard cases (no context)
    for run in range(N_RUNS):
        for chat, label in test_cases:
            futures.append(executor.submit(run_single, chat, label, run))

    # Retraction cases (with context)
    for run in range(N_RUNS):
        for current, context, label in retraction_cases:
            futures.append(executor.submit(run_single, current, label, run, context))

    for future in as_completed(futures):
        label, run_idx, score = future.result()
        results[label]["scores"].append(score.importance)
        results[label]["retractions"].append(score.retraction)
        emo_str = " + ".join(f"{e.emotion.value}({e.weight:.1f})" for e in score.emotions)
        results[label]["emotions_list"].append(emo_str)

# Ordered labels
all_labels = [label for _, label in test_cases] + [label for _, _, label in retraction_cases]
seen = list(dict.fromkeys(all_labels))

for label in seen:
    scores = results[label]["scores"]
    emos = results[label]["emotions_list"]
    retracts = results[label]["retractions"]
    retract_rate = sum(retracts) / len(retracts) if retracts else 0
    retract_tag = f"  retraction={retract_rate:.0%}" if label.startswith("retract") else ""
    print(f"[{label:<24}]  imp={[f'{s:.2f}' for s in scores]}{retract_tag}")
    for emo in emos:
        print(f"  {' ' * 26}{emo}")

In [ ]:
import plotly.graph_objects as go
import numpy as np

# Ordered labels
all_labels = [label for _, label in test_cases] + [label for _, _, label in retraction_cases]
labels = list(dict.fromkeys(all_labels))

means = [np.mean(results[tag]["scores"]) for tag in labels]
stds = [np.std(results[tag]["scores"]) for tag in labels]

# --- Fig 1: Importance mean ± std bar chart (6-tier zones) ---
colors = []
for tag in labels:
    if tag.startswith("trivial"):
        colors.append("#9e9e9e")
    elif tag.startswith("fact"):
        colors.append("#42a5f5")
    elif tag.startswith("light"):
        colors.append("#66bb6a")
    elif tag.startswith("moderate"):
        colors.append("#ffa726")
    elif tag.startswith("deep"):
        colors.append("#ff7043")
    elif tag.startswith("core"):
        colors.append("#ef5350")
    else:
        colors.append("#ab47bc")  # retraction

fig1 = go.Figure()
fig1.add_trace(go.Bar(
    x=labels, y=means,
    error_y=dict(type="data", array=stds, visible=True),
    marker_color=colors, marker_line=dict(color="black", width=0.5),
    showlegend=False,
))

# 6-tier zone shading
for y0, y1, color, label in [
    (0.0, 0.19, "gray", "trivial"),
    (0.2, 0.49, "blue", "factual"),
    (0.5, 0.59, "green", "light pref"),
    (0.6, 0.69, "orange", "moderate"),
    (0.7, 0.79, "red", "deep"),
    (0.8, 1.0, "darkred", "core"),
]:
    fig1.add_hrect(y0=y0, y1=y1, fillcolor=color, opacity=0.06, line_width=0)

fig1.update_layout(
    title=f"Importance Score: Mean ± Std  (N={N_RUNS} runs per case)",
    yaxis_title="Importance", yaxis_range=[0, 1.05],
    xaxis_tickangle=-45, height=500, template="plotly_white",
)
fig1.show()

# --- Fig 2: Weighted emotion heatmap ---
all_emotions = [e.value for e in Emotion]
emotion_matrix = np.zeros((len(labels), len(all_emotions)))

for col, tag in enumerate(labels):
    for emo_str in results[tag]["emotions_list"]:
        for part in emo_str.split(" + "):
            name, w = part.rsplit("(", 1)
            row = all_emotions.index(name)
            emotion_matrix[col][row] += float(w.rstrip(")"))

emotion_matrix /= N_RUNS

z = emotion_matrix.T
text = [[f"{z[r][c]:.2f}" if z[r][c] > 0.05 else ""
         for c in range(z.shape[1])] for r in range(z.shape[0])]

fig2 = go.Figure()
fig2.add_trace(go.Heatmap(
    z=z, x=labels, y=all_emotions,
    colorscale="YlOrRd", zmin=0, zmax=1,
    text=text, texttemplate="%{text}",
    colorbar=dict(title="Avg Weight"),
    hovertemplate="case: %{x}<br>emotion: %{y}<br>weight: %{z:.2f}<extra></extra>",
))
fig2.update_layout(
    title=f"Emotion Weight Distribution (avg over N={N_RUNS} runs)",
    xaxis_tickangle=-45, height=500, template="plotly_white",
)
fig2.show()

# --- Fig 3: Retraction accuracy ---
retract_labels = [label for _, _, label in retraction_cases]
retract_rates = [
    sum(results[tag]["retractions"]) / len(results[tag]["retractions"])
    for tag in retract_labels
]

fig3 = go.Figure()
fig3.add_trace(go.Bar(
    x=retract_labels, y=retract_rates,
    marker_color="#ab47bc",
    text=[f"{r:.0%}" for r in retract_rates], textposition="outside",
))
fig3.add_hline(y=1.0, line=dict(color="green", dash="dash", width=1),
               annotation_text="target: 100%", annotation_position="top left")
fig3.update_layout(
    title=f"Retraction Detection Rate (N={N_RUNS} runs per case)",
    yaxis_title="retraction=true rate", yaxis_range=[0, 1.15],
    xaxis_tickangle=-15, height=350, template="plotly_white",
)
fig3.show()

# --- Stats summary ---
print(f"{'Label':<26} {'Mean':>6} {'Std':>6} {'Min':>6} {'Max':>6}  Dominant Emotions (avg weight)")
print("-" * 100)
for tag in labels:
    scores = results[tag]["scores"]
    col = labels.index(tag)
    top_indices = np.argsort(emotion_matrix[col])[::-1]
    top_str = ", ".join(
        f"{all_emotions[i]}={emotion_matrix[col][i]:.2f}"
        for i in top_indices if emotion_matrix[col][i] > 0.05
    )
    retract_info = ""
    if tag.startswith("retract"):
        rate = sum(results[tag]["retractions"]) / len(results[tag]["retractions"])
        retract_info = f"  retract={rate:.0%}"
    print(f"{tag:<26} {np.mean(scores):>6.2f} {np.std(scores):>6.3f} {min(scores):>6.2f} {max(scores):>6.2f}  {top_str}{retract_info}")